# YOLOv11 Multi-GPU Training on UA-DETRAC
This notebook is designed to be executed on **Kaggle** using the **Dual T4 GPU** environment.

## Prerequisites
1. Enable the **T4 x2** accelerator in the Kaggle notebook settings.
2. Attach the dataset: `bratjay/ua-detrac-orig` to this notebook.

In [ ]:
# 1. Install Dependencies
!pip install ultralytics -q
import ultralytics
from ultralytics import YOLO
import os
import glob
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm
import shutil

ultralytics.checks()

## 2. Dataset Conversion (XML to YOLO Format)
Since the raw UA-DETRAC dataset uses XML annotations, we need to convert them to YOLO format (.txt) before training.

In [ ]:
KAGGLE_INPUT_DIR = "/kaggle/input/ua-detrac-orig"
KAGGLE_WORKING_DIR = "/kaggle/working/ua-detrac-yolo"

os.makedirs(os.path.join(KAGGLE_WORKING_DIR, "images/train"), exist_ok=True)
os.makedirs(os.path.join(KAGGLE_WORKING_DIR, "labels/train"), exist_ok=True)

# Map vehicle classes to standard IDs (0: car, 1: bus, 2: van, 3: others)
CLASS_MAP = {
    "car": 0,
    "bus": 1,
    "van": 2,
    "others": 3
}

def convert_xml_to_yolo(xml_dir, img_dir, out_dir):
    processed_images = 0
    xml_files = glob.glob(os.path.join(xml_dir, "*.xml"))
    print(f"Found {len(xml_files)} annotation files. Starting conversion...")
    
    for xml_path in tqdm(xml_files):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # In UA-DETRAC, one XML represents a whole sequence
        seq_name = root.attrib.get("name")
        
        for frame in root.findall("frame"):
            frame_num = int(frame.attrib.get("num"))
            # DETRAC image naming convention usually img00001.jpg etc.
            img_name = f"img{frame_num:05d}.jpg"
            src_img_path = os.path.join(img_dir, seq_name, img_name)
            
            if not os.path.exists(src_img_path):
                continue
                
            # Image dimensions (DETRAC is 960x540)
            img_w, img_h = 960, 540
            
            yolo_labels = []
            target_list = frame.find("target_list")
            if target_list is not None:
                for target in target_list.findall("target"):
                    box = target.find("box")
                    attribute = target.find("attribute")
                    
                    if box is None or attribute is None:
                        continue
                        
                    veh_type = attribute.attrib.get("vehicle_type")
                    class_id = CLASS_MAP.get(veh_type, 3)
                    
                    left = float(box.attrib.get("left"))
                    top = float(box.attrib.get("top"))
                    width = float(box.attrib.get("width"))
                    height = float(box.attrib.get("height"))
                    
                    # Convert to YOLO (cx, cy, nw, nh)
                    cx = (left + width / 2) / img_w
                    cy = (top + height / 2) / img_h
                    nw = width / img_w
                    nh = height / img_h
                    
                    yolo_labels.append(f"{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            
            if yolo_labels:
                processed_images += 1
                dst_img_name = f"{seq_name}_{img_name}"
                dst_img_path = os.path.join(out_dir, "images/train", dst_img_name)
                dst_lbl_path = os.path.join(out_dir, "labels/train", dst_img_name.replace(".jpg", ".txt"))
                
                # Symlink image to save space
                if not os.path.exists(dst_img_path):
                    os.symlink(src_img_path, dst_img_path)
                
                with open(dst_lbl_path, "w") as f:
                    f.write("\n".join(yolo_labels))

    print(f"\nSuccessfully extracted {processed_images} images with valid vehicle annotations.")

# --- DYNAMIC PATH DISCOVERY ---
XML_DIR = None
IMG_DIR = None

for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith('.xml') for f in files):
        XML_DIR = root
    if any(f.endswith('.jpg') for f in files):
        # The root here will be the sequence folder (e.g. MVI_20011)
        # The IMG_DIR we want is the parent of the sequence folders.
        IMG_DIR = os.path.dirname(root)
        break # Found image dir, we can break

print(f"Dynamically found XML_DIR: {XML_DIR}")
print(f"Dynamically found IMG_DIR: {IMG_DIR}")

if XML_DIR and IMG_DIR and os.path.exists(XML_DIR) and os.path.exists(IMG_DIR):
    convert_xml_to_yolo(XML_DIR, IMG_DIR, KAGGLE_WORKING_DIR)
else:
    print("Dataset paths not found. Please verify the dataset is attached to the notebook.")


In [ ]:
# 3. Create YAML config for YOLO
yaml_content = """
path: /kaggle/working/ua-detrac-yolo
train: images/train
val: images/train  # Using train as val for demonstration, normally split a validation set

names:
  0: car
  1: bus
  2: van
  3: others
"""

yaml_path = os.path.join(KAGGLE_WORKING_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())
    
print(f"Created dataset config at {yaml_path}")

## 4. Multi-GPU Training (DataParallel)
To utilize both T4 GPUs on Kaggle simultaneously, we pass `device=[0, 1]`.
Ultralytics YOLO natively leverages **DataParallel (DP / DDP)** under the hood when multiple device IDs are provided, automatically splitting the batches across both GPUs for faster training.

In [ ]:
# Initialize the YOLOv11 nano model (or yolo11m.pt for medium)
model = YOLO('yolo11n.pt')

# Train the model
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=64,           # Increased batch size to saturate dual T4 GPUs
    device=[0, 1],      # <--- ENABLE DataParallel (Uses both T4 GPUs)
    workers=4,
    project='/kaggle/working/yolo_detrac',
    name='multi_gpu_run'
)

In [ ]:
# 5. Zip and Save Results
!zip -r /kaggle/working/training_results.zip /kaggle/working/yolo_detrac/multi_gpu_run
print("Training complete! Download training_results.zip from the output pane.")

## 6. Illustration of Model Results using Graphs
Once the training completes, the final model is saved inside `/kaggle/working/yolo_detrac/multi_gpu_run/weights/best.pt`.

Ultralytics YOLO automatically generates comprehensive training graphs and performance charts. We can display them inline!

In [ ]:
from IPython.display import Image, display
import os

run_dir = "/kaggle/working/yolo_detrac/multi_gpu_run"

# Display training loss curves and mAP graphs
results_img = os.path.join(run_dir, "results.png")
if os.path.exists(results_img):
    print("Training Convergence & Metrics:")
    display(Image(filename=results_img, width=1000))

# Display the Confusion Matrix
cm_img = os.path.join(run_dir, "confusion_matrix.png")
if os.path.exists(cm_img):
    print("\nConfusion Matrix:")
    display(Image(filename=cm_img, width=800))

# Display sample validation predictions
val_pred_img = os.path.join(run_dir, "val_batch0_pred.jpg")
if os.path.exists(val_pred_img):
    print("\nValidation Predictions:")
    display(Image(filename=val_pred_img, width=1000))
